In [0]:
import csv
import random
from datetime import datetime, timedelta

RANGE_LINHAS = 1000
random.seed(42)

# Define file path
file_path = '/Workspace'+ '/'.join(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split('/')[:-1]) + "/files/sales_orders.csv"

# Define schema (columns)
columns = [
    "order_id",
    "order_item_id",
    "order_date",
    "shipment_date",
    "customer_id",
    "customer_name",
    "customer_email",
    "customer_birthdate",
    "customer_gender",
    "customer_city",
    "customer_state",
    "product_id",
    "product_name",
    "product_category",
    "product_subcategory",
    "brand",
    "unit_price",
    "quantity",
    "discount_value",
    "store_id",
    "store_name",
    "store_city",
    "store_state",
    "payment_method",
    "payment_installments",
    "payment_status"
]

# Some synthetic pools
customer_pool = [
    ("CUST-001", "Ana Souza", "ana.souza@example.com", "1995-03-10", "F", "São Paulo", "SP"),
    ("CUST-002", "Bruno Lima", "bruno.lima@example.com", "1990-07-22", "M", "Rio de Janeiro", "RJ"),
    ("CUST-003", "Carla Mendes", "carla.mendes@example.com", "1988-11-05", "F", "Belo Horizonte", "MG"),
    ("CUST-004", "Diego Santos", "diego.santos@example.com", "1992-01-15", "M", "Curitiba", "PR"),
    ("CUST-005", "Elisa Rocha", "elisa.rocha@example.com", "1998-06-30", "F", "Porto Alegre", "RS"),
    ("CUST-006", "Felipe Costa", "felipe.costa@example.com", "1985-09-09", "M", "Salvador", "BA"),
    ("CUST-007", "Gabriela Nunes", "gabriela.nunes@example.com", "1993-04-18", "F", "Fortaleza", "CE"),
    ("CUST-008", "Henrique Alves", "henrique.alves@example.com", "1987-12-01", "M", "Recife", "PE"),
    ("CUST-009", "Isabela Martins", "isabela.martins@example.com", "1996-08-14", "F", "Campinas", "SP"),
    ("CUST-010", "João Pedro", "joao.pedro@example.com", "1991-02-25", "M", "Niterói", "RJ"),
]

product_pool = [
    ("PROD-1001", "Notebook 14\"", "Informática", "Notebook", "Dell", 3500.00),
    ("PROD-1002", "Notebook 15,6\"", "Informática", "Notebook", "Lenovo", 4200.00),
    ("PROD-1003", "Mouse sem fio", "Informática", "Periféricos", "Logitech", 120.00),
    ("PROD-1004", "Teclado mecânico", "Informática", "Periféricos", "Corsair", 550.00),
    ("PROD-1005", "Monitor 24\"", "Informática", "Monitores", "Samsung", 900.00),
    ("PROD-1006", "Smartphone 128GB", "Smartphones", "Celular", "Samsung", 2500.00),
    ("PROD-1007", "Smartphone 256GB", "Smartphones", "Celular", "Apple", 5200.00),
    ("PROD-1008", "Geladeira Frost Free", "Eletrodomésticos", "Refrigerador", "Brastemp", 3800.00),
    ("PROD-1009", "Máquina de lavar 11kg", "Eletrodomésticos", "Lavadora", "Consul", 2200.00),
    ("PROD-1010", "Cadeira gamer", "Móveis", "Cadeira", "ThunderX3", 1100.00),
    ("PROD-1011", "Mesa para escritório", "Móveis", "Mesa", "Tok&Stok", 700.00),
    ("PROD-1012", "Console de videogame", "Games", "Console", "Sony", 4500.00),
    ("PROD-1013", "Jogo de videogame", "Games", "Jogo", "Ubisoft", 250.00),
    ("PROD-1014", "Fone de ouvido Bluetooth", "Áudio", "Fone", "JBL", 300.00),
    ("PROD-1015", "Soundbar 2.1", "Áudio", "Soundbar", "LG", 1300.00),
]

store_pool = [
    ("STORE-01", "Loja Centro SP", "São Paulo", "SP"),
    ("STORE-02", "Loja Barra RJ", "Rio de Janeiro", "RJ"),
    ("STORE-03", "Loja Savassi BH", "Belo Horizonte", "MG"),
    ("STORE-04", "Loja Batel CTBA", "Curitiba", "PR"),
    ("STORE-05", "Loja Shopping POA", "Porto Alegre", "RS"),
]

payment_methods = ["Cartão de Crédito", "Pix", "Boleto", "Cartão de Débito"]
payment_statuses = ["Aprovado", "Pendente", "Cancelado"]

start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)
days_range = (end_date - start_date).days

rows = []
order_counter = 1

# Generate ~1000 line items with realistic orders (1–5 items each)
while len(rows) < RANGE_LINHAS:
    order_id = f"PO-2024-{order_counter:04d}"
    # random order date
    order_date = start_date + timedelta(days=random.randint(0, days_range))
    shipment_delay = random.randint(0, 5)
    shipment_date = order_date + timedelta(days=shipment_delay)

    # choose a customer and store
    cust = random.choice(customer_pool)
    store = random.choice(store_pool)

    # payment details (per order, reused across items)
    payment_method = random.choice(payment_methods)
    if payment_method == "Cartão de Crédito":
        payment_installments = random.randint(1, 12)
    else:
        payment_installments = 1

    payment_status = random.choices(
        population=payment_statuses,
        weights=[0.8, 0.15, 0.05],
        k=1
    )[0]

    # number of items in this order
    num_items = random.randint(1, 5)
    for item_id in range(1, num_items + 1):
        if len(rows) >= RANGE_LINHAS:
            break

        prod = random.choice(product_pool)
        quantity = random.randint(1, 5)
        unit_price = prod[5]

        gross = unit_price * quantity
        # discount up to 15% of gross, sometimes zero
        if random.random() < 0.7:
            discount_value = round(gross * random.uniform(0.0, 0.15), 2)
        else:
            discount_value = 0.00

        row = {
            "order_id": order_id,
            "order_item_id": item_id,
            "order_date": order_date.strftime("%Y-%m-%d"),
            "shipment_date": shipment_date.strftime("%Y-%m-%d"),
            "customer_id": cust[0],
            "customer_name": cust[1],
            "customer_email": cust[2],
            "customer_birthdate": cust[3],
            "customer_gender": cust[4],
            "customer_city": cust[5],
            "customer_state": cust[6],
            "product_id": prod[0],
            "product_name": prod[1],
            "product_category": prod[2],
            "product_subcategory": prod[3],
            "brand": prod[4],
            "unit_price": f"{unit_price:.2f}",
            "quantity": quantity,
            "discount_value": f"{discount_value:.2f}",
            "store_id": store[0],
            "store_name": store[1],
            "store_city": store[2],
            "store_state": store[3],
            "payment_method": payment_method,
            "payment_installments": payment_installments,
            "payment_status": payment_status
        }
        rows.append(row)

    order_counter += 1

# Write CSV
with open(file_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

len(rows), file_path
